# 02 - Embeddings From Classical NLP To OpenAI

## Learning objectives

- Build bag-of-words vectors to understand sparse text representation.
- Compare sparse lexical similarity with dense embedding similarity.
- Use OpenAI embeddings when credentials are available, with deterministic mock embeddings otherwise.
- Explain why Anthropic/Claude is used for generation and critique here, not first-party embeddings.

## Concept primer

Embeddings turn text into vectors. Once text is a vector, we can use geometry to search: similar vectors represent similar meaning. Before neural embeddings, systems often used sparse count vectors such as bag-of-words or TF-IDF. Those methods are still valuable for teaching because students can inspect every dimension.


## Step 1 - Secure setup


In [ ]:
# This setup cell keeps imports working whether Jupyter starts in the repo root
# or inside the nlp_transformers_embeddings folder.
from pathlib import Path
import os
import sys

CURRENT = Path.cwd()
COURSE_ROOT = CURRENT.parent if CURRENT.name == "nlp_transformers_embeddings" else CURRENT
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dotenv import load_dotenv
load_dotenv(COURSE_ROOT / ".env", override=False)
load_dotenv(COURSE_ROOT / "nlp_transformers_embeddings" / ".env", override=False)

# Security practice: print whether keys are present, never the key values.
print("LIVE_API enabled:", os.getenv("LIVE_API", "false").lower() == "true")
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("ANTHROPIC_API_KEY loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


## Step 2 - Build classical bag-of-words vectors

The vocabulary becomes the coordinate system. Each document becomes a list of counts. This is transparent, but brittle: synonyms and paraphrases may not match if they use different words.


In [ ]:
from nlp_transformers_embeddings.utils.mock_data import SAMPLE_DOCUMENTS
from nlp_transformers_embeddings.utils.retrieval import bag_of_words_vectors, cosine_similarity

texts = [doc.text for doc in SAMPLE_DOCUMENTS]
labels = [doc.title for doc in SAMPLE_DOCUMENTS]
vocabulary, bow_vectors = bag_of_words_vectors(texts)

print("Vocabulary size:", len(vocabulary))
print("First 20 vocabulary terms:", vocabulary[:20])
print("First document vector sample:", bow_vectors[0][:20])

query = "How does attention help transformer models connect tokens?"
_, query_vector = bag_of_words_vectors(texts + [query])
# Rebuild together so the query and documents share the same vocabulary.
doc_vectors = query_vector[:-1]
q_vector = query_vector[-1]

for label, vector in zip(labels, doc_vectors):
    print(f"{cosine_similarity(q_vector, vector):.3f} | {label}")


## What just happened?

Bag-of-words found overlap between query words and document words. This can work for exact terms, but it struggles when the wording changes. Neural embeddings are designed to place related meanings close together even when the words differ.


## Step 3 - OpenAI embeddings with mock fallback

OpenAI provides first-party embedding models. This helper calls OpenAI only when `LIVE_API=true` and `OPENAI_API_KEY` is present. Otherwise it returns deterministic mock vectors so the lesson can continue offline.


In [ ]:
from nlp_transformers_embeddings.utils.openai_client import embed_texts
from nlp_transformers_embeddings.utils.retrieval import rank_chunks, chunk_documents

chunks = chunk_documents(SAMPLE_DOCUMENTS, chunk_size_words=55, overlap_words=10)
chunk_texts = [chunk.text for chunk in chunks]

# Real mode: OpenAI text-embedding model. Mock mode: deterministic hash vectors.
chunk_embeddings = embed_texts(chunk_texts)
query_embedding = embed_texts([query])[0]

results = rank_chunks(query_embedding, chunk_embeddings, chunks, top_k=3)
for item in results:
    print(f"Rank {item.rank} | score={item.score:.3f} | {item.chunk.title}")
    print(item.chunk.text[:220] + "...\n")


## Provider comparison note

Claude is not used as the embedding provider here because Anthropic does not currently offer first-party embeddings. Claude is still highly useful in embedding systems: it can rewrite vague queries, synthesize grounded answers from retrieved passages, critique evidence, and evaluate faithfulness.

Anthropic documentation points developers to external embedding providers such as Voyage AI when vector embeddings are needed.


## Student exercise

Try these queries and compare ranking behavior:

- `How should I protect credentials in notebooks?`
- `What is cosine similarity used for?`
- `Why does RAG need retrieved evidence?`

## Debugging checklist

- If all scores look strange in mock mode, remember mock embeddings preserve pipeline shape, not real semantic quality.
- If live OpenAI embeddings fail, confirm `OPENAI_API_KEY` and `LIVE_API=true`.
- If dimensions mismatch, make sure every text was embedded by the same provider/model.

## Production best practices

- Store the embedding model name with your vector index.
- Re-embed documents when you change embedding models.
- Track source metadata so generated answers can cite retrieved evidence.
- Use eval sets, not one or two happy-path queries, to judge retrieval quality.

## Reflection questions

1. Why might lexical search outperform embeddings for exact IDs or product codes?
2. Why might embeddings outperform lexical search for paraphrased questions?
3. What metadata should be stored beside each vector?
